# Tomato Leaf Disease Detector — Colab Training

Trains a MobileNetV2 classifier in two stages — frozen-backbone head training, then
fine-tuning the top of the backbone — and packages everything the local Streamlit app
needs into a single `artifacts.zip`.

**Before running:** `Runtime → Change runtime type → T4 GPU`. Section 1 checks this.

**You need:** `archive.zip` (the Kaggle *Tomato Disease Multiple Sources* download,
~1.4 GB). Upload it to Google Drive once — re-uploading to Colab every session is slow.

**Run order:** top to bottom. Total time on a T4 is roughly 50–70 min: ~12 min unpacking
and verifying 32k images, ~13 min in section 9, ~20 min in section 9b. Set
`FINE_TUNE = False` in section 9b to stop after stage 1. At the end, download
`artifacts.zip` and unpack it into the repo.

---

### Two caveats worth reading

**The module copies.** Section 4 writes copies of `prepare_data.py`, `dataset.py` and
`model.py` so this notebook runs without cloning the repo. They mirror `src/` in the
repo. **If you change the split logic in either place, change it in both** — the local
Streamlit demo picks its test images using the repo's copy, and if the two disagree,
images the model trained on end up in your local test set and local evaluation quietly
overstates performance. Section 13 saves `split_manifest.json` so the two can be diffed.

**Nothing survives a disconnect except Drive.** All working state lives in `/content`,
which is wiped when the runtime recycles — including `history`, so the curves in section
10 cannot be regenerated without retraining. Checkpoints go to Drive precisely so the
model itself is never the thing you lose.

## 1 — Check the GPU

Stops loudly rather than silently training on CPU, which would take hours instead of minutes.

In [ ]:
import tensorflow as tf

# Colab ships TensorFlow preinstalled and preconfigured against its CUDA build.
# Do NOT pip install tensorflow here -- it will break the GPU stack.
print('TensorFlow', tf.__version__)

gpus = tf.config.list_physical_devices('GPU')
if not gpus:
    raise SystemExit(
        'No GPU detected.\n'
        'Runtime -> Change runtime type -> Hardware accelerator: T4 GPU, then Run all.'
    )
print('GPU:', gpus)
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## 2 — Get `archive.zip`

**Option A (recommended):** upload `archive.zip` to your Google Drive once, then mount.
Set `ARCHIVE_IN_DRIVE` to wherever you put it.

**Option B:** set `USE_DRIVE = False` to upload directly through the browser. This
re-uploads 1.4 GB every session, so only use it once.

In [ ]:
from pathlib import Path

USE_DRIVE = True
ARCHIVE_IN_DRIVE = '/content/drive/MyDrive/archive.zip'  # <-- edit if yours is elsewhere

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    archive = Path(ARCHIVE_IN_DRIVE)
    if not archive.exists():
        raise SystemExit(
            f'{archive} not found. Check the path, or set USE_DRIVE = False to upload.'
        )
else:
    from google.colab import files
    print('Select archive.zip (~1.4 GB, this will take a while)...')
    uploaded = files.upload()
    archive = Path('/content') / next(iter(uploaded))

print('Archive:', archive, f'({archive.stat().st_size / 1e9:.2f} GB)')

## 3 — Extract, and look at what we actually got

Extracts to local `/content/`, **never** to mounted Drive — writing ~20k small files to
Drive is pathologically slow.

The printout here is the ground truth for the class names. Whatever it lists is what the
model will be trained on, and `src/advice.py` in the repo needs an entry for each.

In [ ]:
import zipfile

RAW = Path('/content/raw')

if RAW.exists() and any(RAW.iterdir()):
    print(f'{RAW} already populated, skipping extract.')
else:
    RAW.mkdir(parents=True, exist_ok=True)
    print('Extracting (2-4 min)...')
    with zipfile.ZipFile(archive) as zf:
        zf.extractall(RAW)
    print('Done.')

IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.gif', '.webp'}

def tree(root, prefix='', depth=0, max_depth=2):
    """Print the directory shape with per-folder image counts."""
    if depth > max_depth:
        return
    for child in sorted(p for p in root.iterdir() if p.is_dir()):
        n = sum(1 for p in child.iterdir() if p.suffix.lower() in IMAGE_EXTS)
        label = f'{child.name}/' + (f'  [{n} images]' if n else '')
        print(prefix + label)
        tree(child, prefix + '    ', depth + 1, max_depth)

print(f'\nStructure under {RAW}:')
tree(RAW)

## 4 — Write the shared modules

Mirrors of `src/prepare_data.py`, `src/dataset.py` and `src/model.py` from the repo, so
this notebook needs no GitHub clone. See the caveat at the top of the notebook.

In [ ]:
Path('/content/src').mkdir(exist_ok=True)
import sys
if '/content/src' not in sys.path:
    sys.path.insert(0, '/content/src')

In [ ]:
%%writefile /content/src/prepare_data.py
"""Build data/train|val|test, preserving the archive's own train/valid boundary.

Mirrors src/prepare_data.py in the repo. The split must be deterministic and identical
on both machines: filenames are sorted before splitting, the RNG is seeded per class,
and nothing depends on filesystem iteration order.
"""
import json
import random
import shutil
from pathlib import Path

IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.gif', '.webp'}

# tf.io.decode_image -- which image_dataset_from_directory calls -- only understands
# these four. Pillow reads far more (WebP, TIFF, ...), so a file can pass a naive PIL
# check and still crash training with "Unknown image file format. One of JPEG, PNG,
# GIF, BMP required." This archive really does ship WebP payloads inside .jpg
# filenames, so the check below tests the DECODED FORMAT rather than the extension.
TF_READABLE_FORMATS = {'JPEG', 'PNG', 'GIF', 'BMP'}
VALID_ALIASES = ('valid', 'validation', 'val', 'test')
TRAIN_ALIASES = ('train', 'training')


def is_image(path):
    return path.is_file() and path.suffix.lower() in IMAGE_EXTS


def list_images(directory):
    # sorted() is what makes the split reproducible across machines.
    return sorted((p for p in directory.iterdir() if is_image(p)), key=lambda p: p.name)


def class_dirs(directory):
    subdirs = sorted((p for p in directory.iterdir() if p.is_dir()), key=lambda p: p.name)
    return [d for d in subdirs if any(is_image(p) for p in d.iterdir())]


def find_child(directory, aliases):
    by_lower = {p.name.lower(): p for p in directory.iterdir() if p.is_dir()}
    for alias in aliases:
        if alias in by_lower:
            return by_lower[alias]
    return None


def detect_layout(src):
    """Descend through redundant single-child wrappers, then identify the layout."""
    cursor = src
    for _ in range(4):
        train = find_child(cursor, TRAIN_ALIASES)
        valid = find_child(cursor, VALID_ALIASES)
        if train is not None and valid is not None:
            return 'split', {'train': train, 'valid': valid}
        if class_dirs(cursor):
            return 'flat', {'all': cursor}
        children = [p for p in cursor.iterdir() if p.is_dir()]
        if len(children) != 1:
            break
        cursor = children[0]
    raise SystemExit(f'Could not recognise the dataset layout under {src}.')


def copy_all(images, dest):
    dest.mkdir(parents=True, exist_ok=True)
    for img in images:
        shutil.copy2(img, dest / img.name)
    return len(images)


def inspect_image(path):
    """Classify one file as ("ok" | "transcode" | "broken", detected_format).

    "transcode" means the pixels are fine but the container is one TensorFlow cannot
    read (WebP here) -- recoverable by re-encoding. "broken" means it does not decode.
    """
    try:
        from PIL import Image
    except ImportError:
        return 'ok', None
    try:
        with Image.open(path) as img:
            img.verify()
        with Image.open(path) as img:
            fmt = img.format
            img.convert('RGB').load()
    except Exception:
        return 'broken', None
    return ('ok' if fmt in TF_READABLE_FORMATS else 'transcode'), fmt


def transcode_to_jpeg(path):
    """Re-encode a readable-but-wrong-container image as real JPEG, in place.

    Repairing beats deleting: removing the file would change that split's count and
    make these totals disagree with a previous run's reported numbers.
    """
    try:
        from PIL import Image

        with Image.open(path) as img:
            rgb = img.convert('RGB')
            rgb.load()
        rgb.save(path, format='JPEG', quality=95)
        return True
    except Exception:
        return False


def verify_split(out):
    """Repair or remove unusable images. Returns (removed, transcoded).

    Runs AFTER splitting, deliberately: filtering earlier would change each class's
    count and therefore which files halve() sends to val vs test, silently desyncing
    this split from one an already-trained model was built on.
    """
    removed, transcoded = [], []
    for split in ('train', 'val', 'test'):
        split_dir = out / split
        if not split_dir.is_dir():
            continue
        for path in sorted(split_dir.rglob('*')):
            if not path.is_file():
                continue
            verdict, fmt = inspect_image(path)
            if verdict == 'ok':
                continue
            if verdict == 'transcode' and transcode_to_jpeg(path):
                transcoded.append(path)
                continue
            removed.append(path)
            path.unlink()
    return removed, transcoded


def halve(images, seed, class_name):
    """Deterministic 50/50 (val, test) split, seeded per class."""
    shuffled = list(images)
    random.Random(f'{seed}:{class_name}').shuffle(shuffled)
    cut = len(shuffled) // 2
    return shuffled[:cut], shuffled[cut:]


def print_table(counts):
    classes = sorted(counts)
    width = max((len(c) for c in classes), default=5)
    print()
    print(f"{'class'.ljust(width)}  {'train':>7} {'val':>6} {'test':>6} {'total':>7}")
    print('-' * (width + 30))
    totals = {'train': 0, 'val': 0, 'test': 0}
    for name in classes:
        row = counts[name]
        for split in totals:
            totals[split] += row.get(split, 0)
        line_total = sum(row.get(s, 0) for s in totals)
        print(f"{name.ljust(width)}  {row.get('train', 0):>7} "
              f"{row.get('val', 0):>6} {row.get('test', 0):>6} {line_total:>7}")
    print('-' * (width + 30))
    grand = sum(totals.values())
    print(f"{'TOTAL'.ljust(width)}  {totals['train']:>7} "
          f"{totals['val']:>6} {totals['test']:>6} {grand:>7}")
    print()
    print(f'{len(classes)} classes, {grand} images.')
    if grand:
        smallest = min(classes, key=lambda c: sum(counts[c].values()))
        largest = max(classes, key=lambda c: sum(counts[c].values()))
        lo, hi = sum(counts[smallest].values()), sum(counts[largest].values())
        if lo:
            print(f"Imbalance ratio {hi / lo:.1f}x (largest '{largest}' {hi}, "
                  f"smallest '{smallest}' {lo}) -- compensated with class_weight.")


def prepare(src, out, seed=42, force=False, verify=True):
    src, out = Path(src), Path(out)
    if out.exists() and any(out.iterdir()):
        if not force:
            raise SystemExit(f'{out} exists and is not empty. Pass force=True to rebuild.')
        shutil.rmtree(out)

    layout, roots = detect_layout(src)
    print('Detected layout:', layout)
    for label, path in roots.items():
        print(f'  {label}: {path}')

    counts, test_files = {}, {}

    if layout == 'split':
        for class_dir in class_dirs(roots['train']):
            name = class_dir.name
            counts.setdefault(name, {})['train'] = copy_all(
                list_images(class_dir), out / 'train' / name)
        for class_dir in class_dirs(roots['valid']):
            name = class_dir.name
            val_imgs, test_imgs = halve(list_images(class_dir), seed, name)
            row = counts.setdefault(name, {})
            row['val'] = copy_all(val_imgs, out / 'val' / name)
            row['test'] = copy_all(test_imgs, out / 'test' / name)
            test_files[name] = sorted(p.name for p in test_imgs)
    else:
        print('WARNING: flat layout -- no archive boundary to preserve. Using a seeded')
        print('70/15/15 split. Near-duplicates may span splits and inflate test accuracy.')
        for class_dir in class_dirs(roots['all']):
            name = class_dir.name
            images = list_images(class_dir)
            random.Random(f'{seed}:{name}').shuffle(images)
            n = len(images)
            n_train, n_val = int(n * 0.70), int(n * 0.15)
            parts = {
                'train': images[:n_train],
                'val': images[n_train:n_train + n_val],
                'test': images[n_train + n_val:],
            }
            counts[name] = {s: copy_all(imgs, out / s / name) for s, imgs in parts.items()}
            test_files[name] = sorted(p.name for p in parts['test'])

    excluded = []
    repaired = []
    if verify:
        print('\nVerifying every copied image decodes (a minute or two)...')
        removed, transcoded = verify_split(out)
        if transcoded:
            print(f'Re-encoded {len(transcoded)} image(s) whose real format TensorFlow '
                  'cannot read (WebP data in a .jpg filename):')
            for path in transcoded:
                rel = path.relative_to(out)
                repaired.append(str(rel))
                print(f'  {rel}')
            print('Counts are unaffected -- repaired, not dropped.')
        if removed:
            print(f'Removed {len(removed)} undecodable file(s):')
            for path in removed:
                rel = path.relative_to(out)
                excluded.append(str(rel))
                print(f'  {rel}')
                split, class_name = rel.parts[0], rel.parts[1]
                if class_name in counts and split in counts[class_name]:
                    counts[class_name][split] -= 1
                if split == 'test' and class_name in test_files:
                    test_files[class_name] = [
                        n for n in test_files[class_name] if n != path.name]
            print('Counts below exclude them.')
        if not removed and not transcoded:
            print('All images decode cleanly.')

    print_table(counts)

    manifest = {
        'layout': layout,
        'strategy': 'preserve' if layout == 'split' else 'pooled_70_15_15',
        'seed': seed,
        'verified': verify,
        'excluded_undecodable': excluded,
        # Repaired in place. Separate from exclusions because these changed no count.
        'repaired_transcoded': repaired,
        'class_names': sorted(counts),
        'counts': counts,
        'test_files': test_files,
    }
    out.mkdir(parents=True, exist_ok=True)
    (out / 'split_manifest.json').write_text(json.dumps(manifest, indent=2))
    return manifest

In [ ]:
%%writefile /content/src/dataset.py
"""Dataset loading. Mirrors src/dataset.py in the repo."""
from pathlib import Path

import tensorflow as tf

IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE


def get_datasets(data_dir, image_size=IMAGE_SIZE, batch_size=BATCH_SIZE):
    """Return (train_ds, val_ds, test_ds, class_names).

    class_names is alphabetical (image_dataset_from_directory sorts), and that ordering
    is the contract with inference -- it is saved to class_names.json in section 13 and
    read back by src/infer.py.
    """
    data_dir = Path(data_dir)

    def load(split, shuffle):
        return tf.keras.utils.image_dataset_from_directory(
            data_dir / split,
            image_size=image_size,
            batch_size=batch_size,
            label_mode='int',
            shuffle=shuffle,
            seed=42 if shuffle else None,
        )

    train_ds = load('train', True)
    val_ds = load('val', False)     # fixed order: predictions must align with labels
    test_ds = load('test', False)
    class_names = list(train_ds.class_names)

    # Caching is asymmetric on purpose. image_dataset_from_directory yields decoded
    # float32, so an in-memory .cache() on ~20k training images costs roughly
    # 20000 * 224 * 224 * 3 * 4 bytes ~= 12 GB -- more than a free T4 instance has
    # (~12.7 GB), and the session dies mid-epoch-1 with 'your runtime has crashed'.
    # Do not add .cache() to train. If epochs are I/O-bound, use a disk cache:
    # .cache(filename='/content/tf_cache_train'). val/test are ~10x smaller.
    train_ds = train_ds.prefetch(AUTOTUNE)
    val_ds = val_ds.cache().prefetch(AUTOTUNE)
    test_ds = test_ds.cache().prefetch(AUTOTUNE)

    return train_ds, val_ds, test_ds, class_names


def count_per_class(data_dir, split, class_names):
    root = Path(data_dir) / split
    return [sum(1 for p in (root / n).iterdir() if p.is_file()) for n in class_names]

In [ ]:
%%writefile /content/src/model.py
"""Model definition. Mirrors src/model.py in the repo."""
import tensorflow as tf

IMAGE_SIZE = (224, 224)


def build_model(num_classes, image_size=IMAGE_SIZE):
    """Frozen-backbone MobileNetV2 classifier as one Sequential stack.

    Layer order is load-bearing:
    1. Augmentation runs FIRST, on raw 0-255 pixels. RandomContrast rescales around the
       mean assuming that range; after normalisation to [-1, 1] it distorts rather than
       augments. Do not reorder these two blocks.
    2. Rescaling to [-1, 1] is what MobileNetV2 expects and matches
       mobilenet_v2.preprocess_input. Keeping it inside the model makes the saved .keras
       file self-contained, so infer.py passes a plain 0-255 array.
    """
    base_model = tf.keras.applications.MobileNetV2(
        input_shape=(*image_size, 3), include_top=False, weights='imagenet')
    # Frozen: only the head trains, and the base's BatchNorm layers stay in inference
    # mode -- which is what you want while the head is randomly initialised.
    base_model.trainable = False

    return tf.keras.Sequential([
        tf.keras.layers.Input(shape=(*image_size, 3)),
        # -- augmentation, on 0-255 --
        tf.keras.layers.RandomFlip('horizontal'),
        tf.keras.layers.RandomRotation(0.1),
        tf.keras.layers.RandomContrast(0.1),
        # -- normalise to [-1, 1] --
        tf.keras.layers.Rescaling(scale=1.0 / 127.5, offset=-1.0),
        # -- frozen pretrained backbone --
        base_model,
        # -- classification head --
        tf.keras.layers.GlobalAveragePooling2D(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(num_classes, activation='softmax'),
    ], name='tomato_disease_mobilenetv2')


def compile_model(model, learning_rate=1e-3):
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy'])
    return model

## 5 — Build the splits

Archive `train/` becomes the training set; archive `valid/` is split 50/50 into val and
test. Test images were therefore never in the training pool.

This is deliberately *not* a random 70/15/15 re-shuffle of everything pooled together:
the dataset aggregates multiple sources and contains near-duplicate images, so pooling
puts near-duplicates on both sides of the split and test accuracy ends up measuring
memorisation. Takes 2–3 min.

In [ ]:
from prepare_data import prepare

DATA = Path('/content/data')
SEED = 42

manifest = prepare(RAW, DATA, seed=SEED, force=True)

print()
print('Strategy:', manifest['strategy'])

## 6 — Load the datasets and print the class names

The `class_names` list below is the model's output column order. It gets saved to
`class_names.json` in section 13, so there is nothing to copy by hand — but check that
`src/advice.py` in the repo has an entry for each name.

In [ ]:
from dataset import count_per_class, get_datasets

train_ds, val_ds, test_ds, class_names = get_datasets(DATA)

print()
print(f'{len(class_names)} classes, in model output order:')
for i, name in enumerate(class_names):
    print(f'  [{i}]  {name}')

## 7 — Class weights

Counts are uneven across classes. Without weighting, the model can score well on plain
accuracy while quietly failing the smaller classes.

In [ ]:
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

train_counts = count_per_class(DATA, 'train', class_names)
y_train = np.repeat(np.arange(len(class_names)), train_counts)

weights = compute_class_weight('balanced', classes=np.arange(len(class_names)), y=y_train)
class_weight = {i: float(w) for i, w in enumerate(weights)}

for i, name in enumerate(class_names):
    print(f'  {name:<45} n={train_counts[i]:>5}   weight={class_weight[i]:.3f}')

## 8 — Build and compile

In [ ]:
from model import build_model, compile_model

model = compile_model(build_model(len(class_names)))
model.summary()

trainable = sum(int(np.prod(w.shape)) for w in model.trainable_weights)
total = sum(int(np.prod(w.shape)) for w in model.weights)
print()
print(f'Trainable params: {trainable:,} of {total:,} '
      f'({100 * trainable / total:.1f}%) -- backbone is frozen.')

## 9 — Train

~3–5 min per epoch on a T4. Checkpoints go to Drive when it is mounted, so a
disconnect does not cost you the run.

`EarlyStopping(restore_best_weights=True)` means the model in memory afterwards is the
best one by validation accuracy, not necessarily the last epoch's.

In [ ]:
EPOCHS = 10

OUT = Path('/content/out')
(OUT / 'models').mkdir(parents=True, exist_ok=True)
(OUT / 'reports').mkdir(parents=True, exist_ok=True)

ckpt_dir = Path('/content/drive/MyDrive') if USE_DRIVE else (OUT / 'models')
ckpt_path = ckpt_dir / 'tomato_disease_mobilenetv2.keras'
print('Checkpointing to:', ckpt_path)

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        str(ckpt_path), monitor='val_accuracy', mode='max',
        save_best_only=True, verbose=1),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy', mode='max', patience=3,
        restore_best_weights=True, verbose=1),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    class_weight=class_weight,
    callbacks=callbacks,
)

## 9b — Fine-tune the top of the backbone

Section 9 trains **only** the `Dense` head on frozen ImageNet features, and that plateaus
in the low 80s. The signature is in the section 9 log above: train accuracy stops
climbing, and train loss sits level with validation loss instead of dropping below it.
That is underfitting — the head has taken everything generic ImageNet features have to
offer. More epochs, a wider head, or heavier augmentation will not move it; the last
would make it worse.

This stage unfreezes the top third of MobileNetV2 and continues at a much lower learning
rate, so the convolutional filters can specialise on lesion texture and leaf margin
instead of on ImageNet's objects.

Three details are load-bearing:

- **BatchNorm layers stay frozen.** Unfreezing them lets their moving statistics drift on
  small batches and wrecks the pretrained representation within an epoch. Setting
  `trainable = False` on a BatchNormalization layer also forces inference mode, which is
  exactly what is wanted here.
- **The learning rate drops 100×** (1e-3 → 1e-5). At the stage-1 rate the first few
  gradient updates would destroy the pretrained weights before anything useful is learnt.
- **Stage 2 checkpoints to a separate file.** `tomato_disease_mobilenetv2.keras` on Drive
  is never overwritten, so this stage carries no downside risk — if it fails to beat the
  frozen baseline, the cell reloads stage 1 and says so.

Set `FINE_TUNE = False` to skip. Costs roughly 2× stage 1's per-epoch time, since
gradients now flow through most of the backbone — budget ~15–20 min for 8 epochs.

In [ ]:
FINE_TUNE = True
FINE_TUNE_EPOCHS = 8
UNFREEZE_FRACTION = 0.35   # top 35% of backbone layers -- a fraction, not a hardcoded
                           # index, so this holds whatever MobileNetV2's layer count is
FINE_TUNE_LR = 1e-5        # 100x below stage 1

history_ft = None

if FINE_TUNE:
    stage1_best = max(history.history['val_accuracy'])

    # Find the backbone by structure: it is the one nested model inside the Sequential
    # stack. Fail loudly rather than silently fine-tuning nothing.
    nested = [layer for layer in model.layers if hasattr(layer, 'layers')]
    if len(nested) != 1:
        raise SystemExit(
            f'Expected exactly one nested backbone model, found {len(nested)}: '
            f'{[l.name for l in nested]}. Check build_model() in section 4.')
    base = nested[0]

    trainable_before = sum(int(np.prod(w.shape)) for w in model.trainable_weights)

    base.trainable = True
    unfreeze_from = int(len(base.layers) * (1 - UNFREEZE_FRACTION))
    frozen_bn = 0
    for i, layer in enumerate(base.layers):
        if i < unfreeze_from:
            layer.trainable = False
        # BatchNorm stays frozen at every depth, including above the cut. Letting its
        # moving statistics update here corrupts the pretrained representation within an
        # epoch. trainable=False on a BN layer also forces inference mode -- that is a
        # documented Keras special case, and it is the reason this works.
        if isinstance(layer, tf.keras.layers.BatchNormalization):
            layer.trainable = False
            frozen_bn += 1

    # Recompiling is mandatory -- changes to trainable flags only take effect here.
    model.compile(
        optimizer=tf.keras.optimizers.Adam(FINE_TUNE_LR),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy'])

    trainable_after = sum(int(np.prod(w.shape)) for w in model.trainable_weights)
    total = sum(int(np.prod(w.shape)) for w in model.weights)
    if trainable_after <= trainable_before:
        raise SystemExit(
            'Unfreezing changed nothing -- the backbone is still fully frozen. '
            'Do not run the fit below; it would just repeat stage 1.')

    print(f'Backbone: {base.name}, {len(base.layers)} layers')
    print(f'Unfroze layers {unfreeze_from}-{len(base.layers) - 1}; '
          f'{frozen_bn} BatchNorm layers held frozen at every depth.')
    print(f'Trainable params: {trainable_before:,} -> {trainable_after:,} '
          f'of {total:,} ({100 * trainable_after / total:.1f}%)')
    print(f'Stage 1 best val_accuracy: {stage1_best:.4f}')
    print()

    # A SEPARATE checkpoint file. The stage-1 model on Drive is never overwritten, so
    # this stage cannot cost you the result you already have.
    ft_ckpt = ckpt_dir / 'tomato_disease_mobilenetv2_finetuned.keras'
    print('Checkpointing to:', ft_ckpt)

    history_ft = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=FINE_TUNE_EPOCHS,
        class_weight=class_weight,
        callbacks=[
            tf.keras.callbacks.ModelCheckpoint(
                str(ft_ckpt), monitor='val_accuracy', mode='max',
                save_best_only=True, verbose=1),
            tf.keras.callbacks.EarlyStopping(
                monitor='val_accuracy', mode='max', patience=3,
                restore_best_weights=True, verbose=1),
            tf.keras.callbacks.ReduceLROnPlateau(
                monitor='val_accuracy', mode='max', factor=0.3, patience=2,
                min_lr=1e-7, verbose=1),
        ],
    )

    stage2_best = max(history_ft.history['val_accuracy'])
    print()
    print(f'Stage 1 (frozen)     : {stage1_best:.4f}')
    print(f'Stage 2 (fine-tuned) : {stage2_best:.4f}   ({stage2_best - stage1_best:+.4f})')
    if stage2_best < stage1_best:
        print()
        print('Fine-tuning did not beat the frozen baseline. Reloading stage 1 --')
        print('everything downstream uses that model. Report the frozen numbers, and')
        print('say the fine-tuning attempt did not help; that is a real result.')
        model = tf.keras.models.load_model(ckpt_path)
    else:
        print()
        print('Keeping the fine-tuned model. Everything downstream uses it.')
else:
    print('FINE_TUNE = False -- keeping the frozen-backbone model from section 9.')

## 10 — Training curves

Both stages on one axis, with a dashed line where fine-tuning starts. Expect a visible
step up in validation accuracy just after it.

Reading them: a widening gap between the train and validation lines means overfitting.
Validation sitting *above* train is normal here rather than suspicious — augmentation and
dropout are active during training but not during validation, so training batches are
genuinely harder. In stage 1 the two lines running flat and level is the underfitting
signature that motivates section 9b.

In [ ]:
import matplotlib.pyplot as plt

# Stitch stage 1 and stage 2 into one continuous curve. globals().get() rather than a
# bare name so this still runs if section 9b was skipped.
ft = globals().get('history_ft')

hist = {k: list(v) for k, v in history.history.items()}
stage1_epochs = len(hist['loss'])
if ft is not None:
    for key, values in ft.history.items():
        if key in hist:                      # skips stage-2-only keys like learning_rate
            hist[key].extend(values)

epochs_ran = range(1, len(hist['loss']) + 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for ax, key, title in ((axes[0], 'accuracy', 'Accuracy'), (axes[1], 'loss', 'Loss')):
    ax.plot(epochs_ran, hist[key], marker='o', label='train', color='#B8863B')
    ax.plot(epochs_ran, hist['val_' + key], marker='o', label='validation', color='#1E4635')
    if ft is not None:
        ax.axvline(stage1_epochs + .5, color='#1F2A24', ls='--', lw=1, alpha=.55)
        ax.annotate('fine-tune →', xy=(stage1_epochs + .65, 0.04),
                    xycoords=('data', 'axes fraction'),
                    fontsize=8, color='#1F2A24', alpha=.75)
    ax.set_title(title)
    ax.set_xlabel('epoch')
    ax.grid(alpha=.25)
    ax.legend()
fig.tight_layout()
fig.savefig(OUT / 'reports' / 'training_curves.png', dpi=150)
plt.show()

best = max(hist['val_accuracy'])
print(f'Best validation accuracy overall: {best:.4f} '
      f'(epoch {hist["val_accuracy"].index(best) + 1} of {len(hist["val_accuracy"])})')
if ft is not None:
    s1 = max(history.history['val_accuracy'])
    s2 = max(ft.history['val_accuracy'])
    print(f'  stage 1, frozen backbone : {s1:.4f}')
    print(f'  stage 2, fine-tuned      : {s2:.4f}   ({s2 - s1:+.4f})')
print()
print('Quote this as VALIDATION accuracy. Test accuracy comes from section 11.')

## 11 — Evaluate on the held-out test split

**These are the numbers for your report.** Evaluation runs here rather than only locally,
so the figures do not depend on your laptop reproducing the same split.

Macro F1 leads because the dataset is imbalanced — plain accuracy lets a failing minority
class hide behind the majority ones.

> **If `model.predict` below fails with `InvalidArgumentError: Invalid PNG data`,**
> run the next cell first. A few files in this dataset are truncated or have an
> extension that disagrees with their actual encoding, and `prepare()` copies bytes
> without decoding them, so they survive into the splits.

In [ ]:
# Quarantine images TensorFlow cannot decode.
#
# Only test/ needs scanning: train/ and val/ were fully iterated on every epoch, so
# they are already proven decodable. Deleting from /content/data is safe -- it is
# ephemeral Colab storage rebuilt by re-running section 5, not your archive.
from pathlib import Path

import tensorflow as tf

bad = []
test_root = Path(DATA) / 'test'
for path in sorted(test_root.rglob('*')):
    if not path.is_file():
        continue
    try:
        tf.io.decode_image(tf.io.read_file(str(path)), channels=3, expand_animations=False)
    except Exception as exc:
        bad.append((path, type(exc).__name__))

print(f'Scanned {sum(1 for p in test_root.rglob("*") if p.is_file())} test images.')
if not bad:
    print('All decode cleanly -- the error was something else.')
else:
    print(f'{len(bad)} undecodable:')
    for path, err in bad:
        print(f'  {path.relative_to(test_root)}  ({err})')
        path.unlink()
    print()
    print('Deleted. Re-run section 6 to reload the datasets, then continue from section 11.')
    print('Note the count in your report: these images are excluded from the test set.')

In [ ]:
import json

from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, f1_score)

probs = model.predict(test_ds, verbose=1)
y_pred = probs.argmax(axis=1)
y_true = np.concatenate([y.numpy() for _, y in test_ds])  # valid: test_ds shuffle=False

accuracy = accuracy_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, average='macro')
per_class_f1 = f1_score(y_true, y_pred, average=None, labels=range(len(class_names)))
report = classification_report(y_true, y_pred, target_names=class_names,
                               digits=4, zero_division=0)
cm = confusion_matrix(y_true, y_pred, labels=range(len(class_names)))

print()
print(f'Macro F1 : {macro_f1:.4f}   <- headline metric')
print(f'Accuracy : {accuracy:.4f}')
print(f'Test images: {len(y_true)}')
print()
print(report)

(OUT / 'reports' / 'classification_report.txt').write_text(
    f'Macro F1: {macro_f1:.4f}\nAccuracy: {accuracy:.4f}\n'
    f'Test images: {len(y_true)}\n\n{report}')
(OUT / 'reports' / 'metrics.json').write_text(json.dumps({
    'accuracy': float(accuracy),
    'macro_f1': float(macro_f1),
    'n_test_images': int(len(y_true)),
    'class_names': class_names,
    'per_class_f1': {n: float(s) for n, s in zip(class_names, per_class_f1)},
    'confusion_matrix': cm.tolist(),
}, indent=2))

worst = min(zip(class_names, per_class_f1), key=lambda pair: pair[1])
print(f'Weakest class: {worst[0]} (F1 {worst[1]:.3f}) -- worth a sentence in the report.')

In [ ]:
# Confusion matrix, row-normalised so large classes do not visually swamp small ones.
with np.errstate(invalid='ignore', divide='ignore'):
    norm = np.nan_to_num(cm.astype('float') / cm.sum(axis=1, keepdims=True))

n = len(class_names)
fig, ax = plt.subplots(figsize=(max(8, n * .85), max(7, n * .75)))
im = ax.imshow(norm, cmap='YlGn', vmin=0, vmax=1)
ax.set_xticks(range(n), class_names, rotation=45, ha='right', fontsize=8)
ax.set_yticks(range(n), class_names, fontsize=8)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title('Confusion matrix (row-normalised)')
for i in range(n):
    for j in range(n):
        if cm[i, j]:
            ax.text(j, i, str(cm[i, j]), ha='center', va='center', fontsize=7,
                    color='white' if norm[i, j] > .55 else '#1E4635')
fig.colorbar(im, ax=ax, fraction=.046, pad=.04)
fig.tight_layout()
fig.savefig(OUT / 'reports' / 'confusion_matrix.png', dpi=150)
plt.show()

## 12 — Eyeball a few predictions

Sanity check before leaving Colab. Green = correct, ochre = wrong. Some errors are
expected; what you want to rule out is systematic nonsense, e.g. everything predicted as
one class, which would mean the labels and images are misaligned.

In [ ]:
import numpy as np

# test_ds is deliberately unshuffled, so a single batch would be almost all one class.
# Unbatch and shuffle across the whole split to get a class-diverse sample.
sample = list(test_ds.unbatch().shuffle(4096, seed=0).take(8).as_numpy_iterator())
sample_images = np.stack([img for img, _ in sample])
sample_labels = [int(lbl) for _, lbl in sample]
sample_probs = model.predict(sample_images, verbose=0)

fig, axes = plt.subplots(2, 4, figsize=(15, 8))
for ax, img, true_idx, prob in zip(axes.ravel(), sample_images, sample_labels, sample_probs):
    pred_idx = int(prob.argmax())
    correct = pred_idx == true_idx
    ax.imshow(img.astype('uint8'))
    ax.axis('off')
    ax.set_title(
        f'pred: {class_names[pred_idx]}\n({prob[pred_idx] * 100:.1f}%)\n'
        f'true: {class_names[true_idx]}',
        fontsize=8, color='#1E4635' if correct else '#B8863B')
fig.tight_layout()
plt.show()

print(f'{sum(int(p.argmax()) == t for p, t in zip(sample_probs, sample_labels))}/8 correct '
      'in this sample (too small to mean much -- section 11 has the real numbers).')

## 12b — What the model learned: filters, feature maps, Grad-CAM

Accuracy alone cannot distinguish a model that reads lesions from one that latched onto a
background artefact. These three views make the difference visible, and they mirror
`src/explain.py`, which is what the Streamlit **Explainability** tab renders.

- **First-layer filters** — the 32 kernels of `Conv1`, each a 3x3x3 patch. They sit below
  the unfreeze cut so they are pure ImageNet: generic edge, blob and colour-opponent
  detectors.
- **Feature maps** — activations at three depths for one real test image. Early layers keep
  the leaf outline; the final 7x7 maps encode *what* is present, not what it looks like.
- **Grad-CAM** — the final conv maps weighted by how much each moved the predicted class.
  Bright regions are the pixels that actually drove the decision. If these land on the
  lesion, the score is trustworthy; if they land on the background, it is not.

Everything is saved into `reports/`, so it travels back in `artifacts.zip`.

In [ ]:
# Grad-CAM through a NESTED backbone.
#
# The textbook recipe -- Model(model.input, [conv.output, model.output]) -- does not work
# here: out_relu belongs to the inner MobileNetV2 graph, which the outer Sequential never
# sees, so Keras raises a graph-disconnected error. Instead split the stack into
# (preprocessing, backbone, head) and re-run the segments by hand. Same layers, same
# weights, exact -- and it survives the nesting. src/explain.py does this identically.
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf

REPORTS = OUT / 'reports'
GRADCAM_LAYER = 'out_relu'
FEATURE_LAYERS = ('block_1_expand_relu', 'block_6_expand_relu', 'out_relu')

nested = [l for l in model.layers if hasattr(l, 'layers')]
assert len(nested) == 1, f'expected one nested backbone, found {[l.name for l in nested]}'
backbone = nested[0]
cut = model.layers.index(backbone)
pre_layers, head_layers = model.layers[:cut], model.layers[cut + 1:]
print(f'backbone {backbone.name}: {len(backbone.layers)} layers | '
      f'pre={len(pre_layers)} head={len(head_layers)}')


def apply_pre(x):
    """Augmentation (no-ops at inference) + Rescaling, so callers pass raw 0-255."""
    for layer in pre_layers:
        x = layer(x, training=False)
    return x


def run_head(conv_out):
    y = conv_out
    for layer in head_layers:
        y = layer(y, training=False)
    return y


def load_pixels(path):
    """Disk -> (1, 224, 224, 3) float32 in 0-255, matching infer.preprocess()."""
    raw = tf.io.decode_image(tf.io.read_file(str(path)), channels=3,
                             expand_animations=False)
    resized = tf.image.resize(raw, (224, 224))
    return tf.expand_dims(tf.cast(resized, tf.float32), 0)


def gradcam(x, class_index=None):
    """Returns (heatmap 7x7 in [0,1], class_index, confidence)."""
    feature_model = tf.keras.Model(backbone.inputs,
                                   backbone.get_layer(GRADCAM_LAYER).output)
    with tf.GradientTape() as tape:
        conv_out = feature_model(apply_pre(x), training=False)
        tape.watch(conv_out)
        preds = run_head(conv_out)
        if class_index is None:
            class_index = int(tf.argmax(preds[0]))
        score = preds[:, class_index]
    grads = tape.gradient(score, conv_out)
    weights = tf.reduce_mean(grads, axis=(0, 1, 2))
    heat = tf.nn.relu(tf.reduce_sum(conv_out[0] * weights, axis=-1))
    peak = tf.reduce_max(heat)
    # A flat map means no positive evidence anywhere -- return zeros, do not divide by 0.
    heat = heat / peak if peak > 0 else heat
    return heat.numpy(), int(class_index), float(preds[0][class_index])


# --- 1. first-layer filters --------------------------------------------------
conv1 = next(l for l in backbone.layers if isinstance(l, tf.keras.layers.Conv2D))
kernels = np.transpose(conv1.kernel.numpy(), (3, 0, 1, 2))  # -> (n, h, w, in_ch)
# Per-filter min-max: a global normalisation lets one high-contrast kernel flatten the
# rest into grey mush.
flat = kernels.reshape(len(kernels), -1)
lo = flat.min(axis=1).reshape(-1, 1, 1, 1)
hi = flat.max(axis=1).reshape(-1, 1, 1, 1)
kernels = (kernels - lo) / np.maximum(hi - lo, 1e-8)

fig, axes = plt.subplots(4, 8, figsize=(8, 4.4))
for i, ax in enumerate(axes.ravel()):
    ax.axis('off')
    if i < len(kernels):
        ax.imshow(kernels[i], interpolation='nearest')
        ax.set_title(str(i), fontsize=6, pad=2)
fig.suptitle(f'{conv1.name} — {len(kernels)} filters, {conv1.kernel.shape[:3]} each',
             fontsize=10)
fig.tight_layout()
fig.savefig(REPORTS / 'conv_filters.png', dpi=150, bbox_inches='tight',
            facecolor='white')
plt.show()

# --- 2. pick one real test image per a few classes ---------------------------
test_root = Path(DATA) / 'test'
picks = []
for name in class_names[:4]:
    files = sorted(p for p in (test_root / name).iterdir() if p.is_file())
    if files:
        picks.append((name, files[len(files) // 2]))  # middle file: stable, not cherry-picked
print('using:', [(n, p.name) for n, p in picks])

# --- 3. feature maps at three depths ----------------------------------------
name0, path0 = picks[0]
x0 = load_pixels(path0)
available = [n for n in FEATURE_LAYERS if any(l.name == n for l in backbone.layers)]
extractor = tf.keras.Model(backbone.inputs,
                           [backbone.get_layer(n).output for n in available])
acts = extractor(apply_pre(x0), training=False)
if not isinstance(acts, list):
    acts = [acts]

for layer_name, act in zip(available, acts):
    a = act.numpy()[0]
    # Busiest channels first -- a channel that never fires is a black square.
    energy = a.reshape(-1, a.shape[-1]).mean(axis=0)
    order = np.argsort(energy)[::-1][:16]
    fig, axes = plt.subplots(2, 8, figsize=(12, 3.2))
    for ax, ch in zip(axes.ravel(), order):
        m = a[:, :, ch]
        ax.imshow((m - m.min()) / max(np.ptp(m), 1e-8), cmap='viridis')
        ax.axis('off')
    fig.suptitle(f'{layer_name} — {a.shape} — 16 most active channels ({name0})',
                 fontsize=10)
    fig.tight_layout()
    fig.savefig(REPORTS / f'feature_maps_{layer_name}.png', dpi=150,
                bbox_inches='tight', facecolor='white')
    plt.show()

# --- 4. Grad-CAM across several classes -------------------------------------
fig, axes = plt.subplots(len(picks), 3, figsize=(9, 3.0 * len(picks)))
axes = np.atleast_2d(axes)
for row, (true_name, path) in enumerate(picks):
    x = load_pixels(path)
    heat, idx, conf = gradcam(x)
    base = (x[0].numpy() / 255.0).clip(0, 1)
    big = tf.image.resize(heat[..., None], (224, 224), method='bicubic').numpy()[..., 0]
    big = (big - big.min()) / max(np.ptp(big), 1e-8)

    axes[row, 0].imshow(base)
    axes[row, 0].set_title(f'true: {true_name}', fontsize=9)
    axes[row, 1].imshow(big, cmap='inferno')
    axes[row, 1].set_title('Grad-CAM', fontsize=9)
    axes[row, 2].imshow(base)
    axes[row, 2].imshow(big, cmap='inferno', alpha=0.45)
    mark = 'correct' if class_names[idx] == true_name else 'WRONG'
    axes[row, 2].set_title(f'pred: {class_names[idx]} {conf * 100:.1f}% ({mark})',
                           fontsize=9)
    for ax in axes[row]:
        ax.axis('off')
fig.suptitle(f'Grad-CAM from {GRADCAM_LAYER} — bright = drove the prediction',
             fontsize=11)
fig.tight_layout()
fig.savefig(REPORTS / 'gradcam.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

print()
print('Saved to reports/: conv_filters.png, gradcam.png, and one '
      'feature_maps_<layer>.png per depth.')
print('Read the overlays: heat on the lesion means the score is earned. Heat on the '
      'background or the pot rim means it is not, whatever the accuracy says.')

## 13 — Package everything for download

Produces one `artifacts.zip`. Unpack it at the repo root — the paths inside already match
the repo layout.

In [ ]:
import shutil

# Saves the in-memory model, which is whichever one section 9b left standing: the
# fine-tuned model if it beat the frozen baseline, otherwise the reloaded stage-1 model.
# EarlyStopping(restore_best_weights=True) ran in both stages, so either way this is the
# best-by-val-accuracy version, not merely the last epoch's.
model_path = OUT / 'models' / 'tomato_disease_mobilenetv2.keras'
model.save(model_path)

# The contract that removes any hand-copying of class names.
(OUT / 'models' / 'class_names.json').write_text(json.dumps({
    'class_names': class_names,
    'num_classes': len(class_names),
    'image_size': [224, 224],
}, indent=2))

# Lets you confirm the local split matches this one.
shutil.copy2(DATA / 'split_manifest.json', OUT / 'models' / 'split_manifest.json')

# Record how the model was actually trained, so the README cannot drift from the run.
# globals().get throughout, so this survives section 9b being skipped or deleted.
ft = globals().get('history_ft')
fine_tune_on = globals().get('FINE_TUNE', False)
(OUT / 'reports' / 'training_run.json').write_text(json.dumps({
    'stage1_epochs_ran': len(history.history['loss']),
    'stage1_best_val_accuracy': float(max(history.history['val_accuracy'])),
    'fine_tuned': ft is not None and max(ft.history['val_accuracy'])
                  >= max(history.history['val_accuracy']),
    'stage2_epochs_ran': len(ft.history['loss']) if ft else 0,
    'stage2_best_val_accuracy': float(max(ft.history['val_accuracy'])) if ft else None,
    'unfreeze_fraction': globals().get('UNFREEZE_FRACTION') if fine_tune_on else None,
    'fine_tune_lr': globals().get('FINE_TUNE_LR') if fine_tune_on else None,
}, indent=2))

archive_out = shutil.make_archive('/content/artifacts', 'zip', OUT)
size_mb = Path(archive_out).stat().st_size / 1e6

print(f'Model: {model_path.stat().st_size / 1e6:.1f} MB')
print(f'artifacts.zip: {size_mb:.1f} MB')
print()
print('Contents:')
print('  models/tomato_disease_mobilenetv2.keras')
print('  models/class_names.json')
print('  models/split_manifest.json')
print('  reports/classification_report.txt')
print('  reports/metrics.json')
print('  reports/training_run.json')
print('  reports/confusion_matrix.png')
print('  reports/training_curves.png')
for extra in sorted(p.name for p in (OUT / 'reports').glob('*.png')):
    if extra not in ('confusion_matrix.png', 'training_curves.png'):
        print(f'  reports/{extra}')
print()
print('Unzip at the repo root:  unzip artifacts.zip -d .')
print('Then:                    streamlit run src/streamlit_app.py')

In [ ]:
from google.colab import files

# Also drop a copy in Drive -- browser downloads of ~30 MB occasionally fail.
if USE_DRIVE:
    shutil.copy2(archive_out, '/content/drive/MyDrive/artifacts.zip')
    print('Copied to Drive: MyDrive/artifacts.zip')

files.download(archive_out)